# What Your Reranker Never Sees

## Section 1 - Elastic Serverless Provisioning

In [11]:
%%bash
echo "--- Initializing ---"
terraform -chdir=terraform init -upgrade -input=false > /dev/null && echo "Done."
echo "--- Applying Changes ---"
terraform -chdir=terraform apply -auto-approve > /dev/null && echo "Done."

echo "--- Exporting Environment Variables ---"
cat > .env << EOF
ELASTIC_CLOUD_API_KEY=$(terraform -chdir=terraform output -raw elastic_cloud_api_key)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
EOF
echo "Done."

--- Initializing ---
Done.
--- Applying Changes ---
Done.
--- Exporting Environment Variables ---
Done.


## Section 2 - Connect to the Serverless project


In [12]:
import os
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv(override=True)

CONFIG = {
    "index": "manuals",
    "semantic_index": "manuals_semantic",   # same manuals, content as semantic_text (Section 8)
    "inference_id": ".jina-reranker-v3.5",
    "rank_window_size": 3,          # the three models of one product category
    "max_chunk_size": 300,          # words per chunk for chunk_rescorer
    "chunk_size": 1,                # best-matching chunks sent per manual
    "manuals_dir": "assets/data/manuals",
    "queries_file": "assets/data/queries.json",
}

es = Elasticsearch(
    cloud_id=os.environ["ELASTIC_CLOUD_ID"],
    api_key=os.environ["ELASTIC_CLOUD_API_KEY"],
    request_timeout=300,
)

print(es.info())


{'name': 'serverless', 'cluster_name': 'cf71b90da7a74d938ea41f23aef24ef9', 'cluster_uuid': 'FLtrsGgaTDayExd939mVSw', 'version': {'number': '9.6.0', 'build_flavor': 'serverless', 'build_type': 'docker', 'build_hash': '531321e46298b07b2896d50aed116838c670e60f', 'build_date': '2026-09-04T09:39:50.852949032Z', 'build_snapshot': False, 'lucene_version': '10.5.1', 'minimum_wire_compatibility_version': '9.6.0', 'minimum_index_compatibility_version': '9.6.0'}, 'tagline': 'You Know, for Search'}


## Section 3 - The cap, measured

In [13]:
from chunk_rescorer.probe import probe_doc

QUERY = "What is the maintenance interval for the Kestrel-7 hydraulic actuator?"
MARKER = (
    "The Kestrel-7 hydraulic actuator requires a complete fluid replacement and seal "
    "inspection every 1,450 operating hours according to the manufacturer's service bulletin."
)
DOC_TOKENS = 14_000                                 # well past the ~8,192-token cap
MARKER_OFFSETS = [2_000, 6_000, 7_500, 9_000, 12_000]

control = probe_doc(MARKER, None, DOC_TOKENS)      # same filler, no marker

print(f"{'marker at':>10}  seen?")
for offset in MARKER_OFFSETS:
    resp = es.inference.rerank(
        inference_id=CONFIG["inference_id"],
        query=QUERY,
        input=[control, probe_doc(MARKER, offset, DOC_TOKENS)],   # control first
    )
    scores = {r["index"]: r["relevance_score"] for r in resp["rerank"]}
    seen = scores[1] > scores[0]                    # the copy with the answer must win
    print(f"{offset:>10,}  {'yes' if seen else 'no'}")


 marker at  seen?
     2,000  yes
     6,000  yes
     7,500  yes
     9,000  no
    12,000  no


## Section 4 - Load and index the corpus

In [14]:
from elasticsearch import helpers
from chunk_rescorer.corpus import load_manuals, describe

docs = load_manuals(CONFIG["manuals_dir"])
describe(docs)

mapping = {
    "properties": {
        "content": {"type": "text"},                  # what the reranker reads
        "category": {"type": "keyword"},              # first-stage filter: the shopper's aisle
        "brand": {"type": "keyword"},
        "model": {"type": "keyword"},
        "name": {"type": "text"},
        "token_count": {"type": "integer"},
        "troubleshooting_start_tokens": {"type": "integer"},
    }
}

if not es.indices.exists(index=CONFIG["index"]):
    es.indices.create(index=CONFIG["index"], mappings=mapping)

helpers.bulk(es, ({"_index": CONFIG["index"], "_id": d["model"], **d} for d in docs))
es.indices.refresh(index=CONFIG["index"])

print(f"\nIndexed {es.count(index=CONFIG['index'])['count']} manuals into '{CONFIG['index']}'")


model      category                tokens  troubleshooting at  past the cap?
RG-3200P   pressure washer         23,563              19,618  yes
RG-2600P   pressure washer         20,040              16,287  yes
RG-4000P   pressure washer         15,622              11,670  yes
KG-1200B   garage door opener      34,774              26,047  yes
KG-800C    garage door opener      26,069              20,235  yes
KG-1500W   garage door opener      30,867              25,754  yes
RG-7500E   portable generator      19,435              14,536  yes
RG-4500E   portable generator      21,205              16,670  yes
RG-10000D  portable generator      22,598              18,148  yes
NF-D20B    cordless drill/driver    4,827               3,814  no
KT-7W      smart thermostat         6,077               4,408  no
NF-HR100   hose reel                2,804               2,234  no

12 manuals, ~227,881 tokens in total

Indexed 12 manuals into 'manuals'


## Section 5 - Two retriever configurations

In [15]:
from chunk_rescorer.display import show_comparison


def first_stage(q, category):
    """The manuals of the category the shopper is browsing; the whole catalog if category is None."""
    return {"standard": {"query": {"bool": {
        "filter": [{"term": {"category": category}}] if category else [],
        "must": [{"match": {"content": q}}],
    }}}}


def rerank(q, category, window=CONFIG["rank_window_size"]):
    """Rerank: send each manual whole to the reranker."""
    return {"text_similarity_reranker": {
        "retriever": first_stage(q, category),
        "field": "content",
        "inference_id": CONFIG["inference_id"],
        "inference_text": q,
        "rank_window_size": window,
    }}


def chunk_rerank(q, category, window=CONFIG["rank_window_size"], size=CONFIG["chunk_size"]):
    """Chunk Rerank: send only each manual's best-matching chunk(s)."""
    r = rerank(q, category, window)
    r["text_similarity_reranker"]["chunk_rescorer"] = {
        "size": size,
        "chunking_settings": {"strategy": "sentence", "max_chunk_size": CONFIG["max_chunk_size"], "sentence_overlap": 1},
    }
    return r


query, category, expected = "pressure washer pump drips from weep hole when trigger released", "pressure washer", "RG-3200P"

runs = []
for label, saw, retriever in [
    ("Rerank        whole manuals", "the first ~8,000 tokens of each of the 3 manuals", rerank(query, category)),
    ("Chunk Rerank  best chunk only", "the ~400-token chunk of each manual that best matches the query", chunk_rerank(query, category)),
]:
    resp = es.search(index=CONFIG["index"], retriever=retriever, size=CONFIG["rank_window_size"], source_excludes=["content"])
    runs.append((label, saw, resp))

show_comparison(query, expected, runs)


Query:    "pressure washer pump drips from weep hole when trigger released"
Answer:   RG-3200P is the manual whose troubleshooting section has this fault

Rerank        whole manuals
   reranker saw: the first ~8,000 tokens of each of the 3 manuals
   took:         1,924 ms
   #1   RG-3200P   pressure washer        score 1.148  <- correct
   #2   RG-2600P   pressure washer        score 1.120
   #3   RG-4000P   pressure washer        score 1.100
   verdict:      correct manual first, 0.03 ahead of #2 -> a near tie: the reranker could not really tell them apart

Chunk Rerank  best chunk only
   reranker saw: the ~400-token chunk of each manual that best matches the query
   took:         333 ms
   #1   RG-3200P   pressure washer        score 1.665  <- correct
   #2   RG-4000P   pressure washer        score 1.091
   #3   RG-2600P   pressure washer        score 0.988
   verdict:      correct manual first, 0.57 ahead of #2 -> a clear call

Scores are only comparable within one block, not ac

## Section 6 - Run the queries

In [16]:
from chunk_rescorer.corpus import load_queries
from chunk_rescorer.display import outcome, show_query_table

rows = []
for q in load_queries(CONFIG["queries_file"], kind="symptom"):
    for cfg, retriever in (("Rerank", rerank(q["query"], q["category"])), ("Chunk Rerank", chunk_rerank(q["query"], q["category"]))):
        resp = es.search(index=CONFIG["index"], retriever=retriever, size=CONFIG["rank_window_size"], source_excludes=["content"])
        q[cfg] = outcome(resp, q["target"])
    rows.append(q)

show_query_table(rows)


query                    answer    |         Rerank         |      Chunk Rerank     
                                   |    rank  margin     ms |    rank  margin     ms
------------------------------------------------------------------------------------
pw-inlet-screen          RG-2600P  |      #1   +0.09  1,735 |      #1   +0.43    146
pw-weep-hole             RG-3200P  |      #1   +0.03  1,771 |      #1   +0.57    133
pw-milky-oil             RG-4000P  |      #3   -0.06  1,827 |      #1   +0.64    129
gdo-chain-sag            KG-800C   |      #2   -0.25  1,895 |      #1   +0.37    130
gdo-3-1-pattern          KG-1200B  |      #2   -0.01  1,964 |      #1   +0.45    123
gdo-camera-black         KG-1500W  |      #1   +0.03  1,931 |      #1   +0.60    139
gen-fuel-valve-detent    RG-4500E  |      #3   -0.16  1,737 |      #1   +0.66    139
gen-twistlock-breaker    RG-7500E  |      #3   -0.11  1,766 |      #1   +0.04    129
gen-propane-stall        RG-10000D |      #1   +0.42  1,797 |    

## Section 7 - The request limit

In [17]:
from elasticsearch import BadRequestError
from chunk_rescorer.display import show_comparison, show_rejection

query, expected = "pressure washer pump drips from weep hole when trigger released", "RG-3200P"
catalog = es.count(index=CONFIG["index"])["count"]  # every manual in the store: no category filter

try:
    es.search(index=CONFIG["index"], retriever=rerank(query, None, window=catalog), size=catalog, source_excludes=["content"])
except BadRequestError as e:
    show_rejection("Rerank        whole manuals, whole catalog", f"nothing: {catalog} whole manuals, ~228,000 tokens, refused before inference", e)

resp = es.search(index=CONFIG["index"], retriever=chunk_rerank(query, None, window=catalog), size=catalog, source_excludes=["content"])
show_comparison(query, expected, [("Chunk Rerank  best chunk only, whole catalog", f"one ~400-token chunk from each of the {catalog} manuals", resp)])

Rerank        whole manuals, whole catalog
   reranker saw: nothing: 12 whole manuals, ~228,000 tokens, refused before inference
   status:       400 BadRequestError
   reason:       Request too large; estimated total tokens exceed the maximum limit of
                 200,000. Reduce document count or document sizes.
   verdict:      no ranking at all -> this configuration cannot rerank a window this large

Query:    "pressure washer pump drips from weep hole when trigger released"
Answer:   RG-3200P is the manual whose troubleshooting section has this fault

Chunk Rerank  best chunk only, whole catalog
   reranker saw: one ~400-token chunk from each of the 12 manuals
   took:         276 ms
   #1   RG-3200P   pressure washer        score 1.745  <- correct
   #2   RG-4000P   pressure washer        score 1.057
   #3   RG-2600P   pressure washer        score 0.999
   #4   NF-HR100   hose reel              score 0.935
   #5   KG-1200B   garage door opener     score 0.929
   #6   NF-D20B 

## Section 8 - Does semantic_text avoid this?

In [18]:
semantic_mapping = {"properties": {
    "content": {"type": "semantic_text"},     # default endpoint and default chunking
    "category": {"type": "keyword"},
    "model": {"type": "keyword"},
}}
if not es.indices.exists(index=CONFIG["semantic_index"]):
    es.indices.create(index=CONFIG["semantic_index"], mappings=semantic_mapping)
    helpers.bulk(es, ({"_index": CONFIG["semantic_index"], "_id": d["model"], "content": d["content"], "category": d["category"], "model": d["model"]} for d in docs))
    es.indices.refresh(index=CONFIG["semantic_index"])


def semantic_stage(q, category):
    """Semantic query on the semantic_text field, same category filter as Section 5."""
    return {"standard": {"query": {"bool": {
        "filter": [{"term": {"category": category}}],
        "must": [{"semantic": {"field": "content", "query": q}}],
    }}}}


def on_semantic(retriever, q, category):
    """Swap the reranker's first stage for the semantic one."""
    retriever["text_similarity_reranker"]["retriever"] = semantic_stage(q, category)
    return retriever


semantic_rows = load_queries(CONFIG["queries_file"], kind="symptom")
for q in semantic_rows:
    for cfg, retriever in (
        ("Semantic only", semantic_stage(q["query"], q["category"])),
        ("Rerank", on_semantic(rerank(q["query"], q["category"]), q["query"], q["category"])),
        ("Chunk Rerank", on_semantic(chunk_rerank(q["query"], q["category"]), q["query"], q["category"])),
    ):
        resp = es.search(index=CONFIG["semantic_index"], retriever=retriever, size=CONFIG["rank_window_size"], source=False)
        q[cfg] = outcome(resp, q["target"])

show_query_table(semantic_rows, configs=("Semantic only", "Rerank", "Chunk Rerank"))

query                    answer    |     Semantic only      |         Rerank         |      Chunk Rerank     
                                   |    rank  margin     ms |    rank  margin     ms |    rank  margin     ms
-------------------------------------------------------------------------------------------------------------
pw-inlet-screen          RG-2600P  |      #1   +0.01    254 |      #1   +0.09  1,849 |      #1   +0.43    236
pw-weep-hole             RG-3200P  |      #1   +0.01     76 |      #1   +0.03  1,822 |      #1   +0.57    206
pw-milky-oil             RG-4000P  |      #1   +0.05     81 |      #3   -0.06  1,912 |      #1   +0.64    266
gdo-chain-sag            KG-800C   |      #1   +0.02     87 |      #2   -0.25  2,115 |      #1   +0.37    207
gdo-3-1-pattern          KG-1200B  |      #1   +0.02     81 |      #2   -0.01  1,965 |      #1   +0.45    199
gdo-camera-black         KG-1500W  |      #1   +0.04     86 |      #1   +0.03  2,005 |      #1   +0.60    193
gen-fuel-v

## Section 9 - When the answer is already in view

In [19]:
from chunk_rescorer.corpus import load_queries
from chunk_rescorer.display import outcome, show_query_table

spec_rows = load_queries(CONFIG["queries_file"], kind="spec")   # answered in the specifications, near the front
for q in spec_rows:
    print(f'{q["id"]:18} "{q["query"]}"')
    for cfg, retriever in (("Rerank", rerank(q["query"], q["category"])), ("Chunk Rerank", chunk_rerank(q["query"], q["category"]))):
        resp = es.search(index=CONFIG["index"], retriever=retriever, size=CONFIG["rank_window_size"], source_excludes=["content"])
        q[cfg] = outcome(resp, q["target"])

print()
show_query_table(spec_rows)

pw-spec-4000psi    "which pressure washer is rated 4,000 PSI"
gdo-spec-camera    "garage door opener with built-in camera and battery backup"
gen-spec-propane   "generator that can run on propane"

query                    answer    |         Rerank         |      Chunk Rerank     
                                   |    rank  margin     ms |    rank  margin     ms
------------------------------------------------------------------------------------
pw-spec-4000psi          RG-4000P  |      #1   +0.34  1,800 |      #1   +0.43    124
gdo-spec-camera          KG-1500W  |      #1   +0.06  1,915 |      #3   -0.10    125
gen-spec-propane         RG-10000D |      #1   +0.28  1,761 |      #1   +0.29    131

Rerank: correct manual first in 3 of 3; median Elasticsearch time 1,800 ms
Chunk Rerank: correct manual first in 2 of 3; median Elasticsearch time 125 ms


## Section 10 - Teardown

In [20]:
%%bash
terraform -chdir=terraform destroy -auto-approve > /dev/null && echo "Done."
rm -f .env

Done.
